# Deep dive 13/13 (last one!): `campaign_sends`

Dictionary declares 2,000,000 rows, Source: Internal, Partition: daily. 21 columns.

Dictionary NOT NULL: `send_id`, `send_date`, `process_date`, `campaign_id`, `customer_id`, `send_channel`, `send_status`, `was_delivered`, `had_conversion` -- **9 columns**. `contracts.py`'s current `required` tuple already has **all 9** -- this is the **first table in the whole series where nothing is missing from `required`**, worth confirming rather than assuming.

This table has the one known, already-fixed data bug in the project: `subject` had a literal `"nan"` baked into 38,142 rows upstream (a NULL `promoted_product` name got string-formatted into a template, e.g. `'¡Oferta especial en nan!'`). Silver's `strip_templated_nan()` transform **nulls out the whole `subject`** on any row where "nan" appears as a whole word (word-boundary regex, not a substring match, so it won't misfire on something like "Nantucket"). Section 7 verifies this fix is actually working on the current data, and that it isn't a partial fix.

This table also has a natural **engagement funnel**: `was_delivered` -> `was_opened` -> `was_clicked` -> `had_conversion`, each logically requiring the previous step. Section 9 checks the funnel holds together, and section 10 checks `click_count` is consistent with `was_clicked`.

`open_country` uses the same `country` transform as `ip_country`/`transaction_country` -- section 8 checks it got the same canonicalization treatment.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side

In [ ]:
sample_ids = q("SELECT send_id FROM bronze.campaign_sends ORDER BY send_id LIMIT 20")["send_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

cols = ("send_id, send_date, process_date, campaign_id, customer_id, send_channel, subject, "
        "send_status, was_delivered, was_opened, was_clicked, click_count, had_conversion, "
        "conversion_value, open_country, send_cost")
print("Bronze:")
display(q(f"SELECT {cols} FROM bronze.campaign_sends WHERE send_id IN ({ids_sql}) ORDER BY send_id"))
print("Silver:")
display(q(f"SELECT {cols} FROM silver.fact_campaign_sends WHERE send_id IN ({ids_sql}) ORDER BY send_id"))

## 1. Column inventory -- names AND types, Bronze vs. Silver (all 21 columns)

In [ ]:
expected_silver_columns = [
    "send_id", "send_date", "process_date", "campaign_id", "customer_id", "send_channel",
    "template_used", "subject", "send_status", "was_delivered", "was_opened", "open_date",
    "was_clicked", "click_date", "click_count", "had_conversion", "conversion_date",
    "conversion_value", "open_device", "open_country", "failure_reason", "send_cost",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_campaign_sends'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_campaign_sends:", missing or "none")
print("Extra in silver.fact_campaign_sends:", extra or "none")
print()
print(f"Total expected: {len(expected_silver_columns)}, total actual: {len(actual_silver_columns)}")
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='campaign_sends' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_campaign_sends' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation + dedup

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.campaign_sends")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT send_id) AS n FROM bronze.campaign_sends")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_campaign_sends")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT send_id) AS n FROM silver.fact_campaign_sends")["n"][0]

print(f"bronze.campaign_sends:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct send_id")
print(f"silver.fact_campaign_sends: {silver_total:,} rows, {silver_distinct_pk:,} distinct send_id")
print()
print("Dictionary declares 2,000,000 rows -- matches" if bronze_total == 2_000_000 else f"Dictionary declares 2,000,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 9 dictionary-declared columns

First table in the series where `contracts.py`'s `required` already has all of them -- confirming, not assuming.

In [ ]:
declared_not_null = ["send_id", "send_date", "process_date", "campaign_id", "customer_id",
                      "send_channel", "send_status", "was_delivered", "had_conversion"]
currently_required = {"send_id","send_date","process_date","campaign_id","customer_id",
                       "send_channel","send_status","was_delivered","had_conversion"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.campaign_sends
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_campaign_sends WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

## 4. Nullable columns -- population rate, Bronze vs. Silver (all 12 nullable columns)

In [ ]:
nullable_cols = ["template_used", "subject", "was_opened", "open_date", "was_clicked",
                  "click_date", "click_count", "conversion_date", "conversion_value",
                  "open_device", "open_country", "failure_reason", "send_cost"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.campaign_sends")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.campaign_sends
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_campaign_sends WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

## 5. Domain checks -- `send_channel`, `send_status`

In [ ]:
# send_channel -- dictionary: Email, SMS, Push, WhatsApp, Voice
print("send_channel:")
display(q("SELECT send_channel, count(*) AS n FROM bronze.campaign_sends GROUP BY 1 ORDER BY n DESC"))
# send_status -- dictionary: Sent, Failed, Bounced, Blocked
print("send_status:")
display(q("SELECT send_status, count(*) AS n FROM bronze.campaign_sends GROUP BY 1 ORDER BY n DESC"))

## 6. FK integrity -- `campaign_id`, `customer_id`, reproduced directly

In [ ]:
fk_checks = [
    ("campaign_id", "silver.dim_marketing_campaigns", "campaign_id"),
    ("customer_id", "silver.dim_customers", "customer_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_campaign_sends WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_campaign_sends t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

## 7. `subject`'s `strip_templated_nan()` fix -- verifying it's actually working, and isn't a partial fix

Bronze should still show the literal templated-nan text; Silver should show NULL on those same rows (the whole field is nulled out, not patched), per the transform's documented behavior.

In [ ]:
print("Bronze rows with a literal 'nan' word in subject:")
bronze_nan = q("""
    SELECT count(*) AS n FROM bronze.campaign_sends
    WHERE regexp_matches(subject, '(^|[^A-Za-z])nan([^A-Za-z]|$)')
""")["n"][0]
print(bronze_nan)
print()
print("Sample of the raw broken text:")
display(q_full("""
    SELECT send_id, subject FROM bronze.campaign_sends
    WHERE regexp_matches(subject, '(^|[^A-Za-z])nan([^A-Za-z]|$)')
    LIMIT 5
"""))
print()
print("Of those exact rows, how many are NULL in Silver's subject (the fix), and does any 'nan' text leak through?")
display(q("""
    SELECT
        count(*) AS total_affected_bronze_rows,
        count(*) FILTER (WHERE s.subject IS NULL) AS nulled_in_silver,
        count(*) FILTER (WHERE s.subject IS NOT NULL) AS still_populated_in_silver
    FROM bronze.campaign_sends b
    JOIN silver.fact_campaign_sends s ON s.send_id = b.send_id
    WHERE regexp_matches(b.subject, '(^|[^A-Za-z])nan([^A-Za-z]|$)')
"""))
print()
print("Any remaining 'nan' word anywhere in Silver's subject (should be zero)?")
display(q("""
    SELECT count(*) AS n FROM silver.fact_campaign_sends
    WHERE subject IS NOT NULL AND regexp_matches(subject, '(^|[^A-Za-z])nan([^A-Za-z]|$)')
"""))
print()
print("False-positive check -- any legitimate subject containing 'nan' as a substring but not a whole word (e.g. a hypothetical place name) that got wrongly nulled?")
display(q_full("""
    SELECT b.send_id, b.subject AS bronze_subject, s.subject AS silver_subject
    FROM bronze.campaign_sends b JOIN silver.fact_campaign_sends s ON s.send_id = b.send_id
    WHERE b.subject ILIKE '%nan%' AND NOT regexp_matches(b.subject, '(^|[^A-Za-z])nan([^A-Za-z]|$)')
    LIMIT 10
"""))

## 8. `open_country` -- canonicalization check

Same `country` transform used on `transaction_country`/`ip_country`. Checking whether `open_country` had (and got) the same México/Mexico fix.

In [ ]:
print("Bronze open_country (raw):")
display(q("SELECT open_country, count(*) AS n FROM bronze.campaign_sends GROUP BY 1 ORDER BY n DESC"))
print("Silver open_country (post-canonicalization):")
display(q("SELECT open_country, count(*) AS n FROM silver.fact_campaign_sends GROUP BY 1 ORDER BY n DESC"))

## 9. Engagement funnel -- `was_delivered` -> `was_opened` -> `was_clicked` -> `had_conversion`

Each step should logically require the one before it (can't click if you never opened, can't open if it wasn't delivered).

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE was_opened = TRUE AND was_delivered = FALSE) AS opened_without_delivered,
        count(*) FILTER (WHERE was_clicked = TRUE AND (was_opened IS NULL OR was_opened = FALSE)) AS clicked_without_opened,
        count(*) FILTER (WHERE had_conversion = TRUE AND (was_clicked IS NULL OR was_clicked = FALSE)) AS converted_without_clicked
    FROM silver.fact_campaign_sends
""")

In [ ]:
# Funnel counts and rates by stage, and send_status vs. was_delivered consistency.
print("send_status vs. was_delivered:")
display(q("""
    SELECT send_status, was_delivered, count(*) AS n
    FROM silver.fact_campaign_sends GROUP BY 1, 2 ORDER BY 1, 2
"""))
print()
print("Overall funnel:")
display(q("""
    SELECT
        count(*) AS total_sends,
        count(*) FILTER (WHERE was_delivered) AS delivered,
        count(*) FILTER (WHERE was_opened) AS opened,
        count(*) FILTER (WHERE was_clicked) AS clicked,
        count(*) FILTER (WHERE had_conversion) AS converted
    FROM silver.fact_campaign_sends
"""))

## 10. `click_count` consistency with `was_clicked`

In [ ]:
q("""
    SELECT
        was_clicked,
        count(*) AS n,
        min(click_count) AS min_click_count, max(click_count) AS max_click_count,
        count(*) FILTER (WHERE click_count IS NULL) AS click_count_null,
        count(*) FILTER (WHERE click_count = 0) AS click_count_zero
    FROM silver.fact_campaign_sends
    GROUP BY 1
    ORDER BY 1
""")

## 11. Date sequencing -- `send_date` -> `open_date` -> `click_date` -> `conversion_date`

In [ ]:
q("""
    SELECT
        count(*) FILTER (WHERE open_date IS NOT NULL AND open_date < send_date) AS opened_before_sent,
        count(*) FILTER (WHERE click_date IS NOT NULL AND open_date IS NOT NULL AND click_date < open_date) AS clicked_before_opened,
        count(*) FILTER (WHERE conversion_date IS NOT NULL AND click_date IS NOT NULL AND conversion_date < click_date) AS converted_before_clicked
    FROM silver.fact_campaign_sends
""")

## 12. `conversion_value`/`send_cost` -- range sanity

In [ ]:
q("""
    SELECT
        min(conversion_value) AS min_conv_value, max(conversion_value) AS max_conv_value,
        min(send_cost) AS min_cost, max(send_cost) AS max_cost,
        count(*) FILTER (WHERE conversion_value < 0) AS negative_conv_value,
        count(*) FILTER (WHERE send_cost < 0) AS negative_cost
    FROM silver.fact_campaign_sends
""")

## 13. `send_date`/`process_date` -- cast correctness, rollback check, window check

In [ ]:
q("""
    SELECT
        typeof(send_date) AS send_date_type, typeof(process_date) AS process_date_type,
        min(send_date) AS earliest_send, max(send_date) AS latest_send,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) AS total,
        count(*) FILTER (WHERE process_date <> CAST(send_date AS DATE)) AS process_date_differs
    FROM silver.fact_campaign_sends
    GROUP BY ALL
""")

In [ ]:
q("""
    SELECT
        count(*) FILTER (WHERE send_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE send_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_campaign_sends
""")

## Checklist

- [ ] Column inventory (1): all 21 Silver columns present, types correct
- [ ] Row reconciliation (2): 2,000,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 9 declared columns populated -- confirms `required` is already complete
- [ ] Nullable columns (4): population rates make sense for all 12 nullable columns
- [ ] Domain checks (5): `send_channel`/`send_status` real values
- [ ] FK integrity (6): both FKs reproduced directly
- [ ] `subject` fix (7): `strip_templated_nan()` verified working, no leaks, no false positives
- [ ] `open_country` (8): canonicalization behaves the same way as `transaction_country`/`ip_country`
- [ ] Funnel (9): `was_delivered`/`was_opened`/`was_clicked`/`had_conversion` logically consistent
- [ ] `click_count` (10): consistent with `was_clicked`
- [ ] Date sequencing (11): funnel dates arrive in logical order
- [ ] Numeric ranges (12): `conversion_value`/`send_cost` no negatives
- [ ] Dates (13): rollback rate, window check

In [ ]:
con.close()
print("closed")